# Tier 1 checks: did every event go through WarpX correctly?

Tier 0 showed the BHLUMI sample and the hand-off code are right. This notebook looks at a finished WarpX benchmark
run and checks the bookkeeping: every injected event comes out exactly once, nothing arrives after the run ends,
the vertices, luminosity and weights are the ones WarpX actually used, and the beam kick is purely radial.

Tests (from the compatibility test plan): **C1** nothing lost, **C2** each event once, **C5** run long enough,
**C6** luminous region, **C7** luminosity, **C8** event weight, **E7** radial kick only.

The run must come from `benchmarks/submit_benchmark.sh`, which tags each lepton with its event number
(`--tag-weights`). Colours: **blue = the run**, **orange = the prediction**, grey = reference lines.

In [ ]:
import glob, math, os, re, sys
import numpy as np
import matplotlib.pyplot as plt

REPO = os.environ.get("BHLUMI_WARPX", os.path.abspath(".."))      # the notebook lives in <repo>/notebooks
sys.path[:0] = [REPO, os.path.join(REPO, "benchmarks")]
import bhabha_theory as bt
import wx_bhabha as wx

PS = os.environ.get("PSCRATCH", "/pscratch/sd/a/alkinder")
BENCH = f"{PS}/runs/warpx/bhabha_benchmark"
# The 100k-event run if it has finished, otherwise the 5000-event dbg_blocking run. Set RUN_DIR to override.
RUN_DIR = os.environ.get("T1_RUN") or next(
    (d for d in (f"{BENCH}/headon_round/seed_1", f"{BENCH}/dbg_blocking/seed_1")
     if os.path.isfile(os.path.join(d, "bhabha_injected.npz")) and os.path.isdir(os.path.join(d, "diags"))),
    f"{BENCH}/dbg_blocking/seed_1")

BLUE, ORANGE, GREY, INK = "#2a78d6", "#eb6834", "#8a8984", "#52514e"
plt.rcParams.update({
    "figure.figsize": (7.5, 3.6), "figure.dpi": 110, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": INK,
    "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8,
    "lines.linewidth": 2, "legend.frameon": False, "axes.titlesize": 12,
})
SUMMARY = []
def verdict(test, ok, msg):
    SUMMARY.append((test, ok, msg))
    print(("PASS" if ok else "FAIL") + f"  {test}: {msg}")
print("run:", RUN_DIR)

In [ ]:
inj = np.load(os.path.join(RUN_DIR, "bhabha_injected.npz"))     # what wx_bhabha.py injected
p = bt.Params(os.path.join(RUN_DIR, "inputs_template.txt"))      # the input this run used
log = open(os.path.join(RUN_DIR, "output.txt"), errors="replace").read()
run = bt.load_run(RUN_DIR)                                         # every lepton WarpX wrote out (walls + box)
N = len(inj["ele"])
nt = int(math.ceil(p.stop_time / p.dt))
print(f"{N} events injected; {len(run['ievt'])} leptons read back; {nt} steps of {p.dt:.4g} s")

## C1. Nothing lost

Each event puts one e⁻ and one e⁺ into WarpX. At the end, each lepton has either hit a wall of the box (and was
recorded there) or is still inside. **You should see** both bars reach the orange line exactly: walls + box =
injected, for each species.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 2.4))
for i, name in enumerate(("e⁻ (ele_bhl)", "e⁺ (pos_bhl)")):
    sp = run["species"] == i
    nw, nb_ = int((sp & (run["where"] == 0)).sum()), int((sp & (run["where"] == 1)).sum())
    ax.barh(i, nw, color=BLUE, height=0.55)
    ax.barh(i, nb_, left=nw + N * 0.004, color=BLUE, alpha=0.45, height=0.55)
    ax.text(nw / 2, i, f"at walls {nw}", ha="center", va="center", color="white", fontsize=9)
    ax.text(nw + nb_ / 2, i, f"still in box {nb_}", ha="center", va="center", color=INK, fontsize=9)
ax.axvline(N, color=ORANGE); ax.text(N, 1.45, f" injected {N}", color=ORANGE, va="center")
ax.set_yticks([0, 1]); ax.set_yticklabels(["e⁻", "e⁺"]); ax.set_ylim(-0.5, 1.7); ax.set_xlabel("leptons")
ax.grid(axis="y", visible=False); ax.set_title("Every injected lepton is accounted for", loc="left")
plt.show()
counts = [int((run["species"] == i).sum()) for i in (0, 1)]
verdict("C1", counts == [N, N], f"found e⁻ {counts[0]}, e⁺ {counts[1]} of {N} each")

## C2. Each event comes out exactly once

Every lepton carries its event number as its weight. If an event were lost it would appear 0 times; if MPI ranks
had each injected a copy, it would appear 4 times. **You should see** all events in the "1" column, for both
species.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.0))
labels = ["0 (lost)", "1", "2", "3", "4+"]; x = np.arange(len(labels)); ok = True
for i, (off, alpha, name) in enumerate(((-0.18, 1.0, "e⁻"), (0.18, 0.45, "e⁺"))):
    times = np.bincount(run["ievt"][run["species"] == i], minlength=N)[:N]
    h = [int((times == k).sum()) for k in range(4)] + [int((times >= 4).sum())]
    ok &= h[1] == N
    ax.bar(x + off, h, width=0.34, color=BLUE, alpha=alpha, label=name)
    for xx, hh in zip(x + off, h):
        if hh: ax.text(xx, hh, f"{hh}", ha="center", va="bottom", fontsize=9, color=INK)
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_xlabel("times an event number appears"); ax.set_ylabel("events")
ax.set_title("Each event number appears exactly once per species", loc="left"); ax.legend(fontsize=9)
plt.show()
verdict("C2", ok, "every tag 1..N appears exactly once in ele_bhl and once in pos_bhl")

## C5. The run is long enough

Each event is injected in the WarpX step that contains its vertex time. An event whose vertex time falls after
the last step would never be injected. **You should see** all vertex times well inside the run (between the grey
lines), peaked where the bunch centres cross.

In [ ]:
steps = inj["t"] / p.dt
fig, ax = plt.subplots()
ax.hist(steps, bins=np.linspace(0, nt, 65), color=BLUE, alpha=0.85)
for xv, lab in ((0, "run starts"), (nt, "run ends")):
    ax.axvline(xv, color=GREY, ls="--", lw=1)
    ax.text(xv, 0.95, f" {lab}" if xv == 0 else f"{lab} ", transform=ax.get_xaxis_transform(), ha="left" if xv == 0 else "right", color=GREY)
ax.axvline(p.muz / wx.CLIGHT / p.dt, color=ORANGE, lw=1.5)
ax.text(p.muz / wx.CLIGHT / p.dt, 0.85, " centres cross", color=ORANGE, transform=ax.get_xaxis_transform())
ax.set_xlabel("vertex time [WarpX step]"); ax.set_ylabel("events")
late_log = re.search(r"(\d+) had vertex times after the end", log)
late = int(late_log.group(1)) if late_log else int((steps >= nt).sum())
ax.set_title(f"All {N} vertices fall inside the {nt}-step run", loc="left"); plt.show()
done = re.search(r"\[wx_bhabha\] injected (\d+) events", log)
verdict("C5", late == 0 and (done is None or int(done.group(1)) == N),
        f"{late} events after the end; log reports {done.group(1) if done else '?'} injected")

## C6. The luminous region, as injected

The same check as Tier 0, now on the vertices this run actually used. For two identical round Gaussian bunches,
collisions happen in a Gaussian region σ/√2 wide in every direction. **You should see** each histogram on its
orange curve.

In [ ]:
s2 = math.sqrt(2); tc = p.muz / wx.CLIGHT
panels = [("x [µm]", 1e6 * inj["x"], 1e6 * p.sx / s2, 0.0), ("y [µm]", 1e6 * inj["y"], 1e6 * p.sy / s2, 0.0),
          ("z [mm]", 1e3 * inj["z"], 1e3 * p.sz / s2, 0.0), ("time [step]", steps, p.sz / (s2 * wx.CLIGHT) / p.dt, tc / p.dt)]
fig, axs = plt.subplots(1, 4, figsize=(11, 3.0)); ok = True
for ax, (lab, d, sig, mu0) in zip(axs, panels):
    b_ = np.linspace(mu0 - 4 * sig, mu0 + 4 * sig, 41)
    ax.hist(d, b_, density=True, color=BLUE, alpha=0.85)
    xx = np.linspace(b_[0], b_[-1], 300); ax.plot(xx, np.exp(-0.5 * ((xx - mu0) / sig)**2) / (math.sqrt(2 * math.pi) * sig), color=ORANGE)
    ax.set_xlabel(lab); ax.set_yticks([]); ax.set_title(f"{np.std(d) / sig:.3f} × σ/√2", loc="left", fontsize=10)
    ok &= abs(np.std(d) / sig - 1) < 4 / math.sqrt(2 * len(d))
plt.tight_layout(); plt.show()
verdict("C6", ok, "x, y, z and time widths equal σ/√2 within statistics")

## C7. Luminosity per crossing

For head-on round Gaussian bunches, 𝓛 = N²/(4πσ²). The vertex sampler's value sets the event weights; WarpX's
own `DifferentialLuminosity` diagnostic is an independent measurement from the simulated bunches. **You should
see** both on the orange line, within the ±1 % band.

In [ ]:
L0 = bt.luminosity_per_crossing(p)
rows = [("vertex sampler (wx_bhabha.py)", float(inj["L_bx"]))]
Lw = bt.luminosity_from_reduced(RUN_DIR, p)
if Lw: rows.append(("WarpX DifferentialLuminosity", Lw))
fig, ax = plt.subplots(figsize=(7.5, 0.9 + 0.6 * len(rows)))
ax.axvspan(L0 * 0.99, L0 * 1.01, color=ORANGE, alpha=0.12, lw=0); ax.axvline(L0, color=ORANGE)
for i, (lab, val) in enumerate(rows):
    ax.plot(val, i, "o", color=BLUE, ms=8)
    ax.text(val, i + 0.3, f"{lab}: {100 * (val / L0 - 1):+.2f} %", ha="center", color=INK, fontsize=9)
ax.set_yticks([]); ax.set_ylim(-0.6, len(rows) - 0.2); ax.set_xlabel("luminosity per crossing [m⁻²]")
ax.set_title(f"N²/(4πσ²) = {L0:.4g} m⁻² (orange, band ±1 %)", loc="left"); plt.show()
verdict("C7", all(abs(v / L0 - 1) < 0.01 for _, v in rows), ", ".join(f"{l} {v:.4g}" for l, v in rows))

## C8. Event weight

Each injected event stands for σ·𝓛/N real Bhabhas, so all the weights together must add up to the number of
Bhabhas per crossing in the generation window, σ·𝓛. This is a single-number check, so no plot.

In [ ]:
w = float(inj["weight"]); xs = float(inj["xsec_nb"]); L = float(inj["L_bx"])
expected = xs * wx.NB_TO_M2 * L
print(f"   weight per event   {w:.4g}")
print(f"   × {N} events      = {w * N:.4g} Bhabhas per crossing")
print(f"   σ·𝓛 = {xs:.4g} nb × {L:.4g} m⁻² = {expected:.4g} Bhabhas per crossing")
verdict("C8", abs(w * N / expected - 1) < 1e-9, f"sum of weights / (σ·𝓛) = {w * N / expected:.12f}")

## E7. The kick is radial

A round bunch pulls a lepton straight towards the beam axis, so it should change the lepton's polar angle θ but
not its azimuth φ, and the kick should be the same in every direction around the beam.

**Left:** the mean kick in each quarter of φ, divided by the closed-form prediction. **You should see** four bars
at 1. **Right:** the sideways (azimuthal) kick θ·Δφ next to the radial kick Δθ. **You should see** the sideways
kick centred on zero, while the radial kick is shifted to negative values (towards the axis).

In [ ]:
pred = bt.kick_closed_form(run["theta0"], p, E_lepton_GeV=run["E0"])
q = (np.floor((run["phi0"] + math.pi) / (math.pi / 2)).astype(int)) % 4
qlab = ["−180° to −90°", "−90° to 0°", "0° to 90°", "90° to 180°"]
ratio, err = [], []
for k in range(4):
    m = q == k
    ratio.append(run["dtheta"][m].mean() / pred[m].mean())
    err.append(run["dtheta"][m].std() / math.sqrt(m.sum()) / abs(pred[m].mean()))
ratio, err = np.array(ratio), np.array(err)
dphi = np.angle(np.exp(1j * (run["phi_out"] - run["phi0"])))
side = run["theta0"] * dphi                                  # azimuthal kick expressed as an angle [rad]

fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
ax = axs[0]
ax.axhline(1, color=ORANGE); ax.bar(range(4), ratio, yerr=err, color=BLUE, width=0.6, capsize=4)
ax.set_xticks(range(4)); ax.set_xticklabels(qlab, fontsize=9); ax.set_xlabel("initial azimuth φ")
ax.set_ylabel("kick / prediction"); ax.set_ylim(0, 1.3); ax.set_title("Same kick in every direction", loc="left")
ax = axs[1]
lim = 4 * np.std(run["dtheta"]) * 1e6
b_ = np.linspace(-lim, lim, 81)
ax.hist(1e6 * side, b_, color=BLUE, alpha=0.85, label="sideways θ·Δφ")
ax.hist(1e6 * run["dtheta"], b_, color=BLUE, histtype="step", lw=2, label="radial Δθ")
ax.axvline(0, color=GREY, lw=1, ls="--")
ax.axvline(1e6 * pred.mean(), color=ORANGE, lw=1.5); ax.text(1e6 * pred.mean(), 0.92, "predicted radial mean ", color=ORANGE,
                                                          ha="right", transform=ax.get_xaxis_transform(), fontsize=9)
ax.set_xlabel("kick [µrad]"); ax.set_ylabel("leptons"); ax.legend(fontsize=9, loc="upper right")
ax.set_title("No sideways kick on average", loc="left")
plt.tight_layout(); plt.show()

chi2 = (((ratio - 1) / err)**2).sum()
side_mean, side_err = side.mean(), side.std() / math.sqrt(len(side))
print(f"   kick / prediction by quadrant: {np.round(ratio, 3)} ± {np.round(err, 3)}  (χ² = {chi2:.1f} for 4)")
print(f"   mean sideways kick {1e6 * side_mean:+.3f} ± {1e6 * side_err:.3f} µrad vs mean radial {1e6 * run['dtheta'].mean():+.3f} µrad")
verdict("E7", chi2 < 13.3 and abs(side_mean) < 3 * side_err + 1e-9,
        f"quadrants consistent with 1 (χ² {chi2:.1f}/4); sideways kick {1e6 * side_mean:+.3f} ± {1e6 * side_err:.3f} µrad")

## Summary

In [ ]:
for test, ok, msg in SUMMARY:
    print(f"{'PASS' if ok else 'FAIL'}  {test:3s}  {msg}")
print(f"\n{sum(ok for _, ok, _ in SUMMARY)} of {len(SUMMARY)} passed  ({RUN_DIR})")